# Stage 1

## Load and Inspect data

In [1]:
#Set paths
import os

DATA_DIR = "data"   # change if needed
DB_FILE = "lahman.db"

In [2]:
#List all .csv files
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
csv_files

['AwardsManagers.csv',
 'Managers.csv',
 'AwardsPlayers.csv',
 'Fielding.csv',
 'Salaries.csv',
 'Parks.csv',
 'Schools.csv',
 'People.csv',
 'PitchingPost.csv',
 'Teams.csv',
 'Appearances.csv',
 'AwardsSharePlayers.csv',
 'TeamsFranchises.csv',
 'Batting.csv',
 'ManagersHalf.csv',
 'FieldingOF.csv',
 'Pitching.csv',
 'CollegePlaying.csv',
 'HomeGames.csv',
 'HallOfFame.csv',
 'AwardsShareManagers.csv',
 'BattingPost.csv',
 'TeamsHalf.csv',
 'SeriesPost.csv',
 'FieldingPost.csv',
 'AllstarFull.csv',
 'FieldingOFsplit.csv']

In [3]:
#Create data frames for the tables we're interested in. 
#This will be used to create the tables
import pandas as pd

people = pd.read_csv(os.path.join(DATA_DIR, "People.csv"))
batting = pd.read_csv(os.path.join(DATA_DIR, "Batting.csv"))
pitching = pd.read_csv(os.path.join(DATA_DIR, "Pitching.csv"))
teams = pd.read_csv(os.path.join(DATA_DIR, "Teams.csv"))
salaries = pd.read_csv(os.path.join(DATA_DIR, "Salaries.csv"))

my_tables = [people, batting, pitching, teams, salaries]

In [4]:
#As sanity check, I print the columns for the 5 tables
for table in my_tables:
    print(table.columns)
    print("")

Index(['ID', 'playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCity',
       'birthCountry', 'birthState', 'deathYear', 'deathMonth', 'deathDay',
       'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast',
       'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'bbrefID',
       'finalGame', 'retroID'],
      dtype='str')

Index(['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH',
       'SF', 'GIDP'],
      dtype='str')

Index(['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'W', 'L', 'G', 'GS',
       'CG', 'SHO', 'SV', 'IPouts', 'H', 'ER', 'HR', 'BB', 'SO', 'BAOpp',
       'ERA', 'IBB', 'WP', 'HBP', 'BK', 'BFP', 'GF', 'R', 'SH', 'SF', 'GIDP'],
      dtype='str')

Index(['yearID', 'lgID', 'teamID', 'franchID', 'divID', 'Rank', 'G', 'Ghome',
       'W', 'L', 'DivWin', 'WCWin', 'LgWin', 'WSWin', 'R', 'AB', 'H', '2B',
       '3B', 'HR', 'BB', 'SO', 'SB', 'CS', 

In [5]:
#Check dtypes
people.dtypes

ID                int64
playerID            str
birthYear       float64
birthMonth      float64
birthDay        float64
birthCity           str
birthCountry        str
birthState          str
deathYear       float64
deathMonth      float64
deathDay        float64
deathCountry        str
deathState          str
deathCity           str
nameFirst           str
nameLast            str
nameGiven           str
weight          float64
height          float64
bats                str
throws              str
debut               str
bbrefID             str
finalGame           str
retroID             str
dtype: object

## Creating Database

In [6]:
#Create the SQLite Database
import sqlite3

conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON;")
cursor = conn.cursor()

In [7]:
# Primary keys for each table
pk_map = {
    "People": ["playerID"],
    "Batting": ["playerID", "yearID", "stint"],
    "Pitching": ["playerID", "yearID", "stint"],
    "Teams": ["teamID", "yearID"],
    "Salaries": ["yearID", "teamID", "playerID"]
}

# Foreign keys for tables that need them (corrected to composite keys)
fk_map = {
    "Batting": [("playerID", "People", "playerID"), ("teamID,yearID", "Teams", "teamID,yearID")],
    "Pitching": [("playerID", "People", "playerID"), ("teamID,yearID", "Teams", "teamID,yearID")],
    "Salaries": [("playerID", "People", "playerID"), ("teamID,yearID", "Teams", "teamID,yearID")]
}

# Map pandas dtypes to SQLite types
dtype_map = {
    "str": "TEXT",
    "int64": "INTEGER",
    "float64": "REAL"
}

# List of tables and their dataframes
table_names = ["People", "Batting", "Pitching", "Teams", "Salaries"]

for name, df in zip(table_names, my_tables):
    # Drop table if exists
    cursor.execute(f"DROP TABLE IF EXISTS {name};")
    
    cols = []
    for col, dtype in zip(df.columns, df.dtypes):
        sql_type = dtype_map.get(str(dtype), "TEXT")
        col_quoted = f'"{col}"'  # quote column names
        # single-column PK
        if col in pk_map[name] and len(pk_map[name]) == 1:
            cols.append(f"{col_quoted} {sql_type} PRIMARY KEY")
        else:
            cols.append(f"{col_quoted} {sql_type}")
    
    # Start CREATE TABLE statement
    create_sql = f"CREATE TABLE {name} ({', '.join(cols)}"
    
    # Composite PK if needed
    if len(pk_map[name]) > 1:
        pk_cols = ', '.join(f'"{c}"' for c in pk_map[name])
        create_sql += f", PRIMARY KEY ({pk_cols})"
    
    # Add foreign keys
    if name in fk_map:
        for fk_cols, ref_table, ref_cols in fk_map[name]:
            # handle composite keys split by comma
            fk_cols_list = ', '.join(f'"{c.strip()}"' for c in fk_cols.split(','))
            ref_cols_list = ', '.join(f'"{c.strip()}"' for c in ref_cols.split(','))
            create_sql += f", FOREIGN KEY ({fk_cols_list}) REFERENCES {ref_table}({ref_cols_list})"
    
    create_sql += ");"
    
    cursor.execute(create_sql)
    print(f"Created table {name}")

conn.commit()

Created table People
Created table Batting
Created table Pitching
Created table Teams
Created table Salaries


In [8]:
# Load parent tables first
people.to_sql("People", conn, if_exists="append", index=False)
teams.to_sql("Teams", conn, if_exists="append", index=False)
print("Loaded People and Teams")

# Then load child tables
batting.to_sql("Batting", conn, if_exists="append", index=False)
pitching.to_sql("Pitching", conn, if_exists="append", index=False)
salaries.to_sql("Salaries", conn, if_exists="append", index=False)
print("Loaded Batting, Pitching, Salaries")

Loaded People and Teams
Loaded Batting, Pitching, Salaries


In [9]:
# List of tables and their corresponding DataFrames
tables = ["People", "Batting", "Pitching", "Teams", "Salaries"]
dfs = [people, batting, pitching, teams, salaries]

for name, df in zip(tables, dfs):
    cursor.execute(f"SELECT COUNT(*) FROM {name};")
    count = cursor.fetchone()[0]
    print(f"{name}: SQLite={count}, DataFrame={len(df)}")

People: SQLite=24270, DataFrame=24270
Batting: SQLite=128598, DataFrame=128598
Pitching: SQLite=57630, DataFrame=57630
Teams: SQLite=3614, DataFrame=3614
Salaries: SQLite=26428, DataFrame=26428
